## 1. Setup

In [1]:
import hashlib
import json
import os
import random
import time
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

In [3]:
BASE = "https://api.themoviedb.org/3"
UA = "datacq-m1-tmdb (kseniia.tikhonova@etu.univ-grenoble-alpes.fr)"
POLITE_DELAY = 0.05

DISCOVER = {"vote_count.gte": 50, "include_adult": "false", "include_video": "false", "sort_by": "primary_release_date.asc", "language": "en-US"}
DETAILS = {"append_to_response": "keywords,credits", "language": "en-US"}

CACHE = Path("cache")
CACHE.mkdir(exist_ok=True)

In [3]:
load_dotenv()
AUTH = {"api_key": os.environ["TMDB_API_KEY"].strip()}
HEADERS = {"User-Agent": UA}

## 2. HTTP helpers
`http_get` retries on 429/5xx and network errors with exponential backoff + jitter (or `Retry-After`).
Other errors (401, 404...) are raised at once. Error messages never include the request URL,
because it contains the API key.
`cached_get` saves every response to `cache/`, so a crashed run can be resumed.

In [4]:
def http_get(url, params, attempts=5):
    for k in range(attempts):
        wait = random.randint(0, 2**k)
        try:
            r = requests.get(url, params={**params, **AUTH}, headers=HEADERS, timeout=10)
        except requests.RequestException as exc:
            status = type(exc).__name__
        else:
            status = r.status_code
            if status == 200:
                return r.json()
            if status not in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"{status} for {url}")
            wait = int(r.headers.get("Retry-After", 0)) or wait

        if k + 1 < attempts:
            print(f"  {status} on attempt {k + 1}, sleeping {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"gave up after {attempts} attempts: {url}")

In [5]:
def cache_key(url, params):
    payload = url + json.dumps(params, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


def cached_get(url, params):
    path = CACHE / f"{cache_key(url, params)}.json"
    if path.exists():
        return json.loads(path.read_text())

    body = http_get(url, params)
    path.write_text(json.dumps(body))
    time.sleep(POLITE_DELAY)
    return body

## 3. Step 1 — movie IDs (`/discover/movie`)
One query per year: TMDB returns at most 500 pages (10,000 results) per query.

In [6]:
time_started = time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())

ids, num_found = [], 0
for year in range(2000, 2025):
    params = {**DISCOVER, "primary_release_date.gte": f"{year}-01-01",
              "primary_release_date.lte": f"{year}-12-31"}
    page, got = 1, 0

    while page <= 500:
        p = cached_get(f"{BASE}/discover/movie", {**params, "page": page})
        results = p["results"]
        ids += [m["id"] for m in results]
        got += len(results)
        if not results or got >= p["total_results"]:
            break
        page += 1

    num_found += p["total_results"]
    print(f"{year}: {got} / {p['total_results']} movies")

ids = sorted(set(ids))
print("found:", num_found, "| unique movies:", len(ids))

2000: 416 / 416 movies
2001: 476 / 476 movies
2002: 494 / 494 movies
2003: 535 / 535 movies
2004: 590 / 590 movies
2005: 614 / 615 movies
2006: 775 / 775 movies
2007: 767 / 767 movies
2008: 840 / 840 movies
2009: 913 / 913 movies
2010: 906 / 906 movies
2011: 965 / 965 movies
2012: 987 / 987 movies
2013: 1129 / 1129 movies
2014: 1223 / 1223 movies
2015: 1276 / 1276 movies
2016: 1339 / 1339 movies
2017: 1416 / 1416 movies
2018: 1441 / 1441 movies
2019: 1420 / 1420 movies
2020: 1102 / 1102 movies
2021: 1137 / 1137 movies
2022: 1181 / 1181 movies
2023: 1096 / 1096 movies
2024: 943 / 943 movies
found: 23982 | unique movies: 23981


## 4. Step 2 — details, keywords and credits (`/movie/{id}`)
`append_to_response` gives all three in one request. One row per movie; we keep only
directors and the top-5 cast (IDs and names) to minimize personal data.

In [7]:
rows, failed = [], []
for i, mid in enumerate(ids, 1):
    try:
        d = cached_get(f"{BASE}/movie/{mid}", DETAILS)
    except (requests.HTTPError, RuntimeError) as e:
        failed.append({"tmdb_id": mid, "error": str(e)})
        continue

    directors = [p for p in d["credits"]["crew"] if p["job"] == "Director"]
    cast = sorted(d["credits"]["cast"], key=lambda p: p["order"])[:5]

    rows.append({
        "tmdb_id": d["id"],
        "title": d["title"],
        "release_date": d["release_date"],
        "original_language": d["original_language"],
        "runtime": d["runtime"],
        "budget": d["budget"],
        "revenue": d["revenue"],
        "vote_average": d["vote_average"],
        "vote_count": d["vote_count"],
        "popularity": d["popularity"],
        "genres": [g["name"] for g in d["genres"]],
        "keywords": [k["name"] for k in d["keywords"]["keywords"]],
        "director_ids": [p["id"] for p in directors],
        "director_names": [p["name"] for p in directors],
        "cast_ids": [p["id"] for p in cast],
        "cast_names": [p["name"] for p in cast],
    })
    if i % 500 == 0:
        print(f"{i} / {len(ids)} movies")

print("failed:", len(failed))

500 / 23981 movies
1000 / 23981 movies
1500 / 23981 movies
2000 / 23981 movies
2500 / 23981 movies
3000 / 23981 movies
3500 / 23981 movies
4000 / 23981 movies
4500 / 23981 movies
5000 / 23981 movies
5500 / 23981 movies
6000 / 23981 movies
6500 / 23981 movies
7000 / 23981 movies
7500 / 23981 movies
8000 / 23981 movies
8500 / 23981 movies
9000 / 23981 movies
9500 / 23981 movies
10000 / 23981 movies
10500 / 23981 movies
11000 / 23981 movies
11500 / 23981 movies
12000 / 23981 movies
12500 / 23981 movies
13000 / 23981 movies
13500 / 23981 movies
14000 / 23981 movies
14500 / 23981 movies
15000 / 23981 movies
15500 / 23981 movies
16000 / 23981 movies
16500 / 23981 movies
17000 / 23981 movies
17500 / 23981 movies
18000 / 23981 movies
18500 / 23981 movies
19000 / 23981 movies
19500 / 23981 movies
20000 / 23981 movies
20500 / 23981 movies
21000 / 23981 movies
21500 / 23981 movies
22000 / 23981 movies
22500 / 23981 movies
23000 / 23981 movies
23500 / 23981 movies
failed: 0


In [8]:
movies = pd.DataFrame(rows)
movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
assert movies["tmdb_id"].is_unique
movies.head()

,tmdb_id,title,release_date,original_language,runtime,budget,revenue,vote_average,vote_count,popularity,genres,keywords,director_ids,director_names,cast_ids,cast_names
0,12,Finding Nemo,2003-05-30,en,100,94000000,940335536,7.822,21042,37.2110,"[Animation, Family, Adventure]","[fish, sydney, australia, parent child relatio...",[7],[Andrew Stanton],"[13, 14, 12, 5293, 118]","[Albert Brooks, Ellen DeGeneres, Alexander Gou..."
1,16,Dancer in the Dark,2000-09-01,en,140,12500000,40061153,7.833,2084,7.9174,"[Drama, Crime]","[factory worker, individual, immigrant, blindn...",[42],[Lars von Trier],"[47, 50, 52, 53, 6748]","[Björk, Catherine Deneuve, David Morse, Peter ..."
2,17,The Dark,2005-09-28,en,87,0,6593579,5.700,299,2.8221,"[Horror, Thriller, Mystery]","[sea, wales, child abuse, shepherd, adolescenc...",[43],[John Fawcett],"[49, 48, 55, 51, 54]","[Maria Bello, Sean Bean, Abigail Stone, Richar..."
3,20,My Life Without Me,2003-03-07,en,106,2500000,12300000,6.142,517,3.5378,"[Drama, Romance]","[dying and death, daughter, farewell, night sh...",[90],[Isabel Coixet],"[98, 99, 100, 103, 101]","[Sarah Polley, Amanda Plummer, Scott Speedman,..."
4,22,Pirates of the Caribbean: The Curse of the Bla...,2003-07-09,en,143,140000000,655011224,7.835,22851,28.5465,"[Adventure, Fantasy, Action]","[ship, blacksmith, gold, exotic island, govern...",[1704],[Gore Verbinski],"[85, 118, 114, 116, 1709]","[Johnny Depp, Geoffrey Rush, Orlando Bloom, Ke..."


## 5. Split into relational tables

In [1]:
movies_table = movies.drop(columns=["director_ids", "director_names", "cast_ids", "cast_names"])
movies_table.head()

,tmdb_id,title,release_date,original_language,runtime,budget,revenue,vote_average,vote_count,popularity,genres,keywords
0,12,Finding Nemo,2003-05-30,en,100,94000000,940335536,7.822,21042,37.2110,"[Animation, Family, Adventure]","[fish, sydney, australia, parent child relatio..."
1,16,Dancer in the Dark,2000-09-01,en,140,12500000,40061153,7.833,2084,7.9174,"[Drama, Crime]","[factory worker, individual, immigrant, blindn..."
2,17,The Dark,2005-09-28,en,87,0,6593579,5.700,299,2.8221,"[Horror, Thriller, Mystery]","[sea, wales, child abuse, shepherd, adolescenc..."
3,20,My Life Without Me,2003-03-07,en,106,2500000,12300000,6.142,517,3.5378,"[Drama, Romance]","[dying and death, daughter, farewell, night sh..."
4,22,Pirates of the Caribbean: The Curse of the Bla...,2003-07-09,en,143,140000000,655011224,7.835,22851,28.5465,"[Adventure, Fantasy, Action]","[ship, blacksmith, gold, exotic island, govern..."


In [2]:
def people_rows(ids_col, names_col, role):
    df = movies[["tmdb_id", ids_col, names_col]].explode([ids_col, names_col]).dropna()
    df.columns = ["tmdb_id", "person_id", "name"]
    df["role"] = role
    return df


directors = people_rows("director_ids", "director_names", "director")
cast = people_rows("cast_ids", "cast_names", "cast")
cast["cast_order"] = cast.groupby("tmdb_id").cumcount() + 1

all_credits = pd.concat([directors, cast]).astype({"person_id": "int64", "cast_order": "Int64"})

people = all_credits[["person_id", "name"]].drop_duplicates("person_id").sort_values("person_id", ignore_index=True)

credits = (all_credits.drop_duplicates(["tmdb_id", "person_id", "role"])
           [["tmdb_id", "person_id", "role", "cast_order"]]
           .sort_values(["tmdb_id", "role", "cast_order"], ignore_index=True))
credits.head()

,tmdb_id,person_id,role,cast_order
0,12,13,cast,1
1,12,14,cast,2
2,12,12,cast,3
3,12,5293,cast,4
4,12,118,cast,5


In [3]:
assert movies_table["tmdb_id"].is_unique
assert people["person_id"].is_unique
assert all_credits.groupby("person_id")["name"].nunique().max() == 1
assert credits["person_id"].isin(people["person_id"]).all()

print("movies:", movies_table.shape, "| people:", people.shape, "| credits:", credits.shape)

movies: (23981, 12) | people: (57029, 2) | credits: (143613, 4)


## 6. Save as Parquet

In [4]:
Path("tables").mkdir(exist_ok=True)
files = {
    "movies.parquet": movies,
    "tables/movies.parquet": movies_table,
    "tables/people.parquet": people,
    "tables/credits.parquet": credits,
}

parquet_shape = {}
for path, df in files.items():
    df.to_parquet(path, index=False, compression="zstd")
    parquet_shape[path] = list(pd.read_parquet(path).shape)
    assert parquet_shape[path] == list(df.shape)
    print(path, parquet_shape[path])

movies.parquet [23981, 16]
tables/movies.parquet [23981, 12]
tables/people.parquet [57029, 2]
tables/credits.parquet [143613, 4]


## 7. Acquisition log

In [5]:
num_found = 23982
unique_movies = 23981
failed = []
time_started = "2026-10-04 20:57:39 UTC"
time_ended = "2026-10-04 21:00:42 UTC"
parquet_shape = {
    "movies.parquet": [23981, 16],
    "tables/movies.parquet": [23981, 12],
    "tables/people.parquet": [57029, 2],
    "tables/credits.parquet": [143613, 4],
}

log = {
    "url": {"discover": f"{BASE}/discover/movie", "details": f"{BASE}/movie/{{id}}"},
    "parameters": {
        "discover": {**DISCOVER, "primary_release_date": "one query per year, 2000-2024"},
        "details": DETAILS,
    },
    "auth": "TMDB api_key (read from .env)",
    "num_found": num_found,
    "time_started": time_started,
    "time_ended": time_ended,
    "unique_movies": unique_movies,
    "failed": failed,
    "parquet_shape": parquet_shape,
    "path_to_file": "movies.parquet",
    "path_to_tables": "tables/",
    "path_to_cache": "cache/",
}
Path("acquisition_log.json").write_text(json.dumps(log, indent=2))
print(json.dumps(log, indent=2))

{
  "url": {
    "discover": "https://api.themoviedb.org/3/discover/movie",
    "details": "https://api.themoviedb.org/3/movie/{id}"
  },
  "parameters": {
    "discover": {
      "vote_count.gte": 50,
      "include_adult": "false",
      "include_video": "false",
      "sort_by": "primary_release_date.asc",
      "language": "en-US",
      "primary_release_date": "one query per year, 2000-2024"
    },
    "details": {
      "append_to_response": "keywords,credits",
      "language": "en-US"
    }
  },
  "auth": "TMDB api_key (read from .env)",
  "num_found": 23982,
  "time_started": "2026-10-04 20:57:39 UTC",
  "time_ended": "2026-10-04 21:00:42 UTC",
  "unique_movies": 23981,
  "failed": [],
  "parquet_shape": {
    "movies.parquet": [
      23981,
      16
    ],
    "tables/movies.parquet": [
      23981,
      12
    ],
    "tables/people.parquet": [
      57029,
      2
    ],
    "tables/credits.parquet": [
      143613,
      4
    ]
  },
  "path_to_file": "movies.parquet",
  